# 50 · The primary pathway list, its coordinate-free support and its external replication

Draft v0.4 makes the **22 pathways robust under both the scFates (SCF13) and the diffusion-pseudotime
(DPT13) coordinates** the primary list. This notebook turns that list into a saved artifact and adds
three checks:

| Part | Question |
|---|---|
| 1 | One row per primary pathway: robust/core under each coordinate, category, direction, peaks in segment units, probe sensitivity, and calls by both step-screen designs on our cohort |
| 2 | Do the 22 (and the 12 core) replicate in donor-level external data beyond random gene sets with the same sizes, strata and overlaps, and beyond relabel-called pathways? |
| 3 | Do the conventional-only pathways replicate their average difference better than other pathways once overlap is respected (v0.1 minor item 20)? |

The protocol was written before running (workstream 5 plan, round 3). Cohort: 2 male control mice
and 2 cortex sections from 1 male human donor; external donors are the replicates in parts 2 and 3.

## 1 · Folders and pre-specified rules

| Rule | Value |
|---|---|
| Primary list | robust under SCF13 and DPT13 (`pt_revision_labels/coordinate_robustness.csv`); core = core under both |
| Guards, part 1 | 22 pathways; 12 core; categories 13/3/1/5; coordinate-free step screen 50 calls on our cohort, 20 of 22 including all 12 core, exceptions probe-sensitive; spline-based steps 8 of 22 |
| Part 2 | notebook 48's overlap-preserving null (9,999 within-strata gene relabelings applied to every set at once); Census and Lake cortex against male mice; primary = signed statistic, matched strata |
| Part 2 decision | specific: p_random ≤ 0.01, p_relabel ≤ 0.05 and program-level label permutation ≤ 0.05 · better than random only: p_random ≤ 0.01 alone · not replicated otherwise |
| Part 3 | reproduce notebook 37's 184 conventional-only pathways and its Mann–Whitney p (guard: within a factor of 2); then mean z of the 184 against random collections, and D = mean(184) − mean(others) with joint relabeling; the group claim stands if D's p ≤ 0.05 in Census |

In [ ]:
import argparse
import json
import os
import sys
from pathlib import Path

start = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
project = next(p for p in (start, *start.parents) if (p / 'pseudospace').is_dir())
sys.path.insert(0, str(project))
parser = argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root', type=Path)
parser.add_argument('--results-root', type=Path)
args, _ = parser.parse_known_args()
data_root = Path(args.data_root or os.environ.get('PSEUDOSPACE_DATA_ROOT') or project / 'data').expanduser()
results_root = Path(args.results_root or os.environ.get('PSEUDOSPACE_RESULTS_ROOT') or project / 'results').expanduser()
upstream13 = results_root / 'minimal_pt_scfates'
upstream37 = results_root / 'pt_pathway_final' / 'scfates' / 'tables'
upstream37_cache = results_root / 'pt_pathway_final' / 'scfates' / 'stage_cache'
upstream43 = results_root / 'pt_revision_classes' / 'reviewed' / 'tables'
upstream45 = results_root / 'pt_revision_method'
upstream47 = results_root / 'pt_revision_addenda' / 'tables'
coordinate_file = results_root / 'pt_revision_labels' / 'coordinate_robustness.csv'
external = data_root / 'external'
output = results_root / 'pt_primary_pathways'
figures, tables = output / 'figures', output / 'tables'
for folder in (figures, tables):
    folder.mkdir(parents=True, exist_ok=True)
lake_file = external / 'kpmp_lake2023' / 'lake2023_snRNA_v1.5.h5ad'
required = [upstream13 / 'cross_species_pt_scfates.h5ad', upstream13 / 'ortholog_map_used.csv',
            results_root / 'pt_pathway_remodeling_scfates' / 'gene_statistics.csv',
            upstream37 / 'table_S2_pathways.csv', upstream37 / 'gene_level_statistics.csv',
            upstream37 / 'matched_and_joint_tests_all_partitions.csv', upstream37 / 'external_average_level_conventional_only.csv',
            upstream45 / 'tables' / 'anchor_our_cohort.csv', upstream47 / 'robust_peaks_in_segment_units.csv', coordinate_file,
            external / 'census_pt_segments' / 'human_pt_segment_pseudobulk_counts.csv.gz',
            external / 'census_pt_segments' / 'mouse_pt_segment_pseudobulk_counts.csv.gz', lake_file]
for path in required:
    if not path.is_file():
        raise FileNotFoundError(f'Missing input: {path}')

NOTEBOOK_LOGIC_VERSION = '50.primary_pathways.1'  # Bump when a cached calculation changes.
ALPHA = .05
SEED, N_PERM = 50, 9999
GSEA_SEED = 12                 # notebook 37
MIN_CELLS_EXTERNAL = 50       # notebook 37
CONTRIBUTING_QUANTILE, PROGRAM_CUT = .9, .5
print('Results folder:', output)

## 2 · Our structures, genes and pathways (notebook 37's construction)

In [ ]:
import anndata as ad
import numpy as np
import pandas as pd
from scipy import sparse
from IPython.display import display
from pseudospace.pathway_inputs import rebuild_pt_expression
from pseudospace.pathway_remodeling import gene_covariates, matching_strata
from pseudospace.pathways import build_pathway_membership
from pseudospace.stage_cache import cached_frame, cached_payload, digest
import pseudospace.set_collection_null as collection_module
import pseudospace.specimen_null as specimen_null_module

pd.set_option('display.width', 220, 'display.max_columns', 30, 'display.max_colwidth', 70)
cache = output / 'stage_cache'
code = digest([NOTEBOOK_LOGIC_VERSION, Path(collection_module.__file__), Path(specimen_null_module.__file__)])
saved_pt = ad.read_h5ad(upstream13 / 'cross_species_pt_scfates.h5ad', backed='r')
pt_all = saved_pt.obs.copy()
saved_pt.file.close()
pt_all = pt_all[pt_all.broad_tubule_marker_call.astype(str).eq('PT')].copy()
expected = {'mouse': ['Ctrl1A2', 'Ctrl1A4'], 'human': ['HUK1_COR1', 'HUK1_MED1']}
names = expected['mouse'] + expected['human']
orthologs = pd.read_csv(upstream13 / 'ortholog_map_used.csv')
source_paths = {name: data_root / 'tubule_by_gene' / f'{name}_tubule_by_gene_caleb.h5ad' for name in names}

support = pt_all.groupby('sample', observed=True).shared_pseudospace.quantile([.01, .99]).unstack()
coordinate = pt_all.shared_pseudospace.to_numpy(float)
pt_obs = pt_all[(coordinate >= support[.01].max()) & (coordinate <= support[.99].min())].copy()
adata = rebuild_pt_expression(pt_obs, source_paths, orthologs, target_sum=1e4)
position = adata.obs.shared_pseudospace.to_numpy(float)
human = adata.obs.comparison_species.astype(str).eq('human').to_numpy()
specimen = adata.obs['sample'].astype(str).to_numpy()
segment = adata.obs.segment_class.astype(str).to_numpy()
expression = sparse.csr_matrix(adata.layers['lognorm'], dtype=float)
universe = gene_covariates(expression, position, human, specimen, adata.var_names)
universe['measured_in_both'] = adata.var.measured_in_both_inputs.to_numpy()
universe['eligible'] = universe.measured_in_both & universe.detection.ge(.02)
genes = universe.loc[universe.eligible, 'gene'].tolist()
saved_genes = pd.read_csv(results_root / 'pt_pathway_remodeling_scfates' / 'gene_statistics.csv', index_col=0).index.tolist()
if genes != saved_genes:
    raise ValueError('The gene universe differs from notebooks 12 and 37.')
Y = expression[:, universe.eligible.to_numpy()]
covariates = universe.set_index('gene').loc[genes]
libraries = ('Reactome_2022', 'MSigDB_Hallmark_2020', 'KEGG_2019_Mouse')
coverage37 = pd.concat([build_pathway_membership(
    json.loads((data_root / 'mouse_vs_human' / 'pathway_gene_sets' / f'{library}.json').read_text()),
    universe.loc[universe.measured_in_both, 'gene'], ortholog_map=orthologs, library_name=library,
    tested=genes, min_genes=1) for library in libraries], ignore_index=True)
coverage37['pathway_id'] = coverage37.library + '::' + coverage37.pathway
coverage37 = coverage37[coverage37.n_tested.between(10, min(300, len(genes) - 1))]
gene_sets = {row.pathway_id: list(row.genes_present) for row in coverage37.itertuples()}
pathway_names = pd.Series({p: p.split('::', 1)[1] for p in gene_sets}).str.replace(r' R-HSA-\d+$', '', regex=True)

table_s2 = pd.read_csv(upstream37 / 'table_S2_pathways.csv', index_col=0)
robust_ids = table_s2.index[table_s2.robust.astype(bool)].tolist()
core_ids = table_s2.index[table_s2.core.astype(bool)].tolist()
T_spatial = pd.read_csv(upstream37 / 'gene_level_statistics.csv', index_col=0).T_spatial.reindex(genes)
tests37 = pd.read_csv(upstream37 / 'matched_and_joint_tests_all_partitions.csv')
spatial37 = tests37[tests37.statistic.eq('T_spatial')]
swaps = [p for p in spatial37.partition.unique() if p.startswith('swap')]
relabel_all = sorted(set(spatial37.loc[spatial37.partition.isin(swaps) & spatial37.effect.gt(0) & spatial37.q_normal.le(ALPHA), 'pathway_id']))
relabel_only = [p for p in relabel_all if p not in set(robust_ids)]
print(f'{len(genes):,} genes · {len(gene_sets):,} pathways · robust {len(robust_ids)} · core {len(core_ids)} · '
      f'relabel-called {len(relabel_all)} ({len(relabel_only)} not robust)')

## 3 · The 22 primary pathways

Peaks in segment units come from notebook 47: 0–1 is S1, 1–2 S2 and 2–3 S3, using pooled
segment-transition positions. Probe-sensitive means the pathway loses significance when only genes
with equal probe counts in both panels are kept (notebook 37's `probe_equal_pass`).

In [ ]:
coord = pd.read_csv(coordinate_file).set_index('pathway_id')
primary_ids = coord.index[coord['robust SCF13'].astype(bool) & coord['robust DPT13'].astype(bool)].tolist()
primary_ids = [p for p in gene_sets if p in set(primary_ids)]   # notebook 37's pathway order
primary_core = [p for p in primary_ids if coord.loc[p, 'core SCF13'] and coord.loc[p, 'core DPT13']]
peaks = pd.read_csv(upstream47 / 'robust_peaks_in_segment_units.csv', header=[0, 1], index_col=0)
peaks.index.name = 'pathway_id'
missing_peaks = [p for p in primary_ids if p not in peaks.index]
if len(primary_ids) != 22 or len(primary_core) != 12 or missing_peaks:
    raise ValueError(f'Expected 22 primary and 12 core pathways with peaks; got {len(primary_ids)}, {len(primary_core)}, {missing_peaks}.')
primary = pd.DataFrame(index=pd.Index(primary_ids, name='pathway_id'))
primary['name'] = table_s2.loc[primary_ids, 'name']
primary['library'] = table_s2.loc[primary_ids, 'library']
for label in ('SCF13', 'DPT13', 'SCF13-ED'):
    primary[f'robust {label}'] = coord.loc[primary_ids, f'robust {label}'].astype(bool)
    primary[f'core {label}'] = coord.loc[primary_ids, f'core {label}'].astype(bool)
primary['core (SCF13 and DPT13)'] = primary.index.isin(primary_core)
primary['notebook 43 category'] = coord.loc[primary_ids, 'notebook 43 category']
for label in ('SCF13', 'DPT13'):
    primary[f'direction {label}'] = np.where(coord.loc[primary_ids, f'human-high at peak {label}'].astype(bool), 'human-high', 'mouse-high')
for label, name in (('scFates', 'SCF13'), ('DPT13', 'DPT13'), ('SCF13-ED', 'SCF13-ED')):
    primary[f'peak {name} (coordinate)'] = pd.to_numeric(peaks.loc[primary_ids, (label, 'peak')])
    primary[f'peak {name} (segment units)'] = pd.to_numeric(peaks.loc[primary_ids, (label, 'segment units (pooled transitions)')])
primary['probe-sensitive'] = ~table_s2.loc[primary_ids, 'probe_equal_pass'].astype(bool)
category_counts = primary['notebook 43 category'].value_counts().to_dict()
expected_categories = {'summary of mouse zonation': 13, 'shared': 3, 'mixed': 1, 'unresolved': 5}
if category_counts != expected_categories:
    raise ValueError(f'Category counts {category_counts} differ from the draft (13/3/1/5).')
if not (primary['direction SCF13'] == primary['direction DPT13']).all():
    raise ValueError('Direction at the peak differs between SCF13 and DPT13.')
print(f'{len(primary)} primary pathways, {len(primary_core)} core; categories {category_counts}; '
      f'{int(primary["probe-sensitive"].sum())} probe-sensitive; {int(primary["direction SCF13"].eq("human-high").sum())} human-high')

## 4 · Coordinate-free step screen on our cohort (notebook 45's code), per pathway

This is notebook 45's anchor for our cohort, recomputed so that every pathway's statistic is saved.
- Structures are pooled to specimen × segment cells.
- The test is the species × segment step (`T_steps`) beyond a species offset, with segment
  intercepts as the shared shape.
- The matched rank-AUC uses exact moments; the joint test divides z by the √VIF of a segment-centred
  CAMERA estimate.
- Calls are BH q ≤ 0.05 with a positive effect.

Notebook 37's spline-based step design (`T_discrete_spatial`) is read from its saved tests.

In [ ]:
from scipy.stats import norm
from statsmodels.stats.multitest import multipletests
from pseudospace.pathway_remodeling import residual_pathway_correlations, correlation_adjusted_rank_tests
from pseudospace.specimen_null import cell_summaries, matched_auc_null, summary_partial_f

# Notebook 45's pathway family: membership built on the universe genes, in build_pathway_membership's order.
family45 = pd.concat([build_pathway_membership(
    json.loads((data_root / 'mouse_vs_human' / 'pathway_gene_sets' / f'{library}.json').read_text()),
    genes, ortholog_map=orthologs, library_name=library, tested=genes, min_genes=1) for library in libraries], ignore_index=True)
family45['pathway_id'] = family45.library + '::' + family45.pathway
family45 = family45[family45.n_tested.between(10, min(300, len(genes) - 1))]
sets45 = {row.pathway_id: list(row.genes_present) for row in family45.itertuples()}
if sorted(sets45) != sorted(gene_sets):
    raise ValueError('Notebook 45 and notebook 37 pathway families differ.')


def cohort_step_screen():
    measured = adata.var.measured_in_both_inputs.to_numpy()
    counts_all = sparse.csr_matrix(adata.layers['counts'], dtype=float)[:, measured]
    panel = adata.var_names[measured][np.asarray(counts_all.sum(axis=0)).ravel() > 0]
    counts_panel = counts_all[:, adata.var_names[measured].get_indexer(panel)]
    library_size = np.asarray(counts_panel.sum(axis=1)).ravel()
    lognorm = counts_panel.multiply((1e4 / library_size)[:, None]).tocsr()
    lognorm.data = np.log1p(lognorm.data)
    universe_ = [g for g in genes if g in set(panel)]
    cols = pd.Index(panel).get_indexer(universe_)
    x, c = lognorm[:, cols], counts_panel[:, cols]
    seg = pd.Series(segment).str.replace('PT-', '', regex=False).to_numpy()
    per_donor = {}
    for name in names:
        rows_ = x[specimen == name]
        seg_d = seg[specimen == name]
        detected = np.array([np.asarray((rows_[seg_d == s] > 0).sum(axis=0)).ravel() > 0 for s in np.unique(seg_d)])
        per_donor[name] = (np.asarray(rows_.mean(axis=0)).ravel(), np.asarray((rows_ > 0).mean(axis=0)).ravel(), detected.mean(axis=0))
    balanced = [np.mean([np.mean([per_donor[d][k] for d in expected[s]], axis=0) for s in ('mouse', 'human')], axis=0) for k in range(3)]
    cov_ = pd.DataFrame({'mean_expression': balanced[0], 'detection': balanced[1], 'coverage': balanced[2]}, index=universe_)
    tested_ = cov_.index[cov_.detection.ge(.02)].tolist()
    if tested_ != genes:
        raise ValueError('Notebook 45\'s cohort pool keeps the whole notebook 37 universe; this one does not.')
    sets_ = {p: v for p, v in sets45.items()}
    summary = cell_summaries(x, specimen, seg, c, library_size)
    corr = []
    for name in names:
        rows_ = x[specimen == name].toarray()
        seg_d = seg[specimen == name]
        for s in np.unique(seg_d):
            rows_[seg_d == s] -= rows_[seg_d == s].mean(axis=0)
        corr.append(residual_pathway_correlations(rows_, np.repeat(name, len(rows_)), tested_, sets_))
    vif = correlation_adjusted_rank_tests(cov_.mean_expression, sets_, pd.concat(corr, ignore_index=True)).set_index('pathway_id').variance_inflation
    group = {d: float(d in expected['human']) for d in names}
    f = summary_partial_f(summary, names, group)
    stats_ = matched_auc_null(pd.DataFrame({'T_steps': f['T_steps']}, index=genes), sets_, matching_strata(cov_, bins=3)).set_index('pathway_id')
    stats_['vif'] = vif.reindex(stats_.index)
    stats_['z_joint'] = stats_.z_matched / np.sqrt(stats_.vif)
    stats_['p_joint'] = norm.sf(stats_.z_joint)
    stats_['q_joint'] = multipletests(stats_.p_joint, method='fdr_bh')[1]
    stats_['called'] = stats_.effect.gt(0) & stats_.q_joint.le(ALPHA)
    return stats_.drop(columns='statistic').reset_index()


steps45 = cached_frame('cohort_step_screen', cohort_step_screen, root=cache,
                       inputs={'structures': list(adata.obs_names), 'genes': digest(genes), 'sets': digest(sorted(sets45.items()))},
                       code=code).set_index('pathway_id')
steps45['name'] = pathway_names.reindex(steps45.index)
steps45.sort_values('p_joint').to_csv(tables / 'cohort_step_screen_per_pathway.csv')
expected_calls = int(pd.read_csv(upstream45 / 'tables' / 'anchor_our_cohort.csv', index_col=0).loc['P1 T_steps joint', 'species'])
cached45 = []
for path in sorted((upstream45 / 'stage_cache').glob('anchor_our_cohort__*.csv')):
    frame = pd.read_csv(path)
    row = frame[frame.kind.eq('summary') & frame.strategy.eq('P1 T_steps joint') & frame.label.eq('species')]
    if len(row) and int(row.calls.iloc[0]) == expected_calls:
        cached45.append({list(sets45)[int(i)] for i in str(row.called.iloc[0]).split('|') if i not in ('', 'nan')})
called45 = set(steps45.index[steps45.called])
if len(called45) != expected_calls or not cached45 or any(c != called45 for c in cached45):
    raise ValueError(f'Recomputed step screen ({len(called45)} calls) does not match notebook 45 ({expected_calls}).')
species37 = tests37[tests37.partition.eq('species')]
spline_steps = set(species37.loc[species37.statistic.eq('T_discrete_spatial') & species37.effect.gt(0) & species37.q_joint.le(ALPHA), 'pathway_id'])
primary['coordinate-free step screen (notebook 45)'] = primary.index.isin(called45)
primary['step screen q (notebook 45)'] = steps45.q_joint.reindex(primary.index)
primary['spline-based steps (notebook 37)'] = primary.index.isin(spline_steps)
exceptions = primary.index[~primary['coordinate-free step screen (notebook 45)']]
checks_ = {'coordinate-free calls among the 22': int(primary['coordinate-free step screen (notebook 45)'].sum()),
           'coordinate-free calls among the 12 core': int(primary.loc[primary_core, 'coordinate-free step screen (notebook 45)'].sum()),
           'exceptions probe-sensitive': bool(primary.loc[exceptions, 'probe-sensitive'].all()),
           'spline-based calls among the 22': int(primary['spline-based steps (notebook 37)'].sum())}
if checks_ != {'coordinate-free calls among the 22': 20, 'coordinate-free calls among the 12 core': 12,
               'exceptions probe-sensitive': True, 'spline-based calls among the 22': 8}:
    raise ValueError(f'Step-screen guards failed: {checks_}')
print('Guards passed:', checks_, '· exceptions:', list(primary.loc[exceptions, 'name']))
primary.to_csv(tables / 'primary_pathway_list.csv')
display(primary[['name', 'core (SCF13 and DPT13)', 'notebook 43 category', 'direction SCF13', 'peak SCF13 (segment units)',
                 'peak DPT13 (segment units)', 'probe-sensitive', 'coordinate-free step screen (notebook 45)',
                 'spline-based steps (notebook 37)']].round(2))

## 5 · Overlap-preserving replication of the 22 and the 12 core

Programs are notebook 37's 17, restricted to the 22 primary pathways; relabel-called pathways are
grouped by the same rule.

In [ ]:
from pseudospace.pathway_decoys import overlap_programs

contributing = set(T_spatial.index[T_spatial.ge(T_spatial.quantile(CONTRIBUTING_QUANTILE))])
robust_programs = table_s2.loc[robust_ids, 'program'].astype(int)
in_order = lambda ids: [p for p in gene_sets if p in set(ids)]   # notebook 37 clusters in pathway order
check_programs = overlap_programs(gene_sets, in_order(robust_ids), contributing=contributing, cut=PROGRAM_CUT).set_index('pathway_id').program
same_partition = pd.crosstab(robust_programs, check_programs.reindex(robust_ids)).gt(0)
if not (same_partition.sum(axis=0).eq(1).all() and same_partition.sum(axis=1).eq(1).all()):
    raise ValueError('The program rule does not reproduce notebook 37\'s 17 robust programs.')
relabel_programs = overlap_programs(gene_sets, in_order(relabel_only), contributing=contributing, cut=PROGRAM_CUT).set_index('pathway_id').program
print(f'{robust_programs.nunique()} robust programs · {relabel_programs.nunique()} relabel-called programs')

primary_programs = robust_programs.loc[primary_ids]
print(f'{primary_programs.nunique()} programs among the 22 primary pathways')

### External donor-level references (notebook 37's inputs)

In [ ]:
def census_donors(label, segment_map, early_parts):
    table = pd.read_csv(external / 'census_pt_segments' / f'{label}_pt_segment_pseudobulk_counts.csv.gz', index_col=0)
    meta = pd.read_csv(external / 'census_pt_segments' / f'{label}_pt_segment_pseudobulk_meta.csv', index_col=0)
    meta = meta[~meta.development_stage.str.contains('embryo|E16|Theiler', case=False)].copy()
    meta['segment'] = meta.cell_type.map(segment_map)
    meta = meta[meta.segment.notna() & meta.n_cells.ge(MIN_CELLS_EXTERNAL)]
    needed = set(segment_map.values())
    complete = meta.groupby('donor_id').segment.apply(lambda s: needed.issubset(set(s)))
    meta = meta[meta.donor_id.isin(complete[complete].index)]
    late, early, whole, info = {}, {}, {}, []

    def log_cpm_of(index):
        return np.log2(table.loc[index].sum() / meta.loc[index, 'total_counts_all_genes'].sum() * 1e6 + 1)

    for (donor, sex), block in meta.groupby(['donor_id', 'sex']):
        late_index, early_index = block.index[block.segment.eq('S3')], block.index[block.segment.isin(early_parts)]
        late[donor], early[donor] = log_cpm_of(late_index), log_cpm_of(early_index)
        whole[donor] = log_cpm_of(late_index.union(early_index))
        info.append({'donor': donor, 'sex': sex})
    frame = lambda values: pd.DataFrame(values).T.reindex(columns=genes)
    return frame(late), frame(early), frame(whole), pd.DataFrame(info).set_index('donor')


h_late, h_early, h_whole, _ = census_donors('human', {'kidney proximal convoluted tubule epithelial cell': 'early',
                                             'epithelial cell of proximal tubule segment 3': 'S3'}, ['early'])
m_late, m_early, m_whole, m_info = census_donors('mouse', {'epithelial cell of proximal tubule segment 1': 'S1',
                                                  'epithelial cell of proximal tubule segment 2': 'S2',
                                                  'epithelial cell of proximal tubule segment 3': 'S3'}, ['S1', 'S2'])
males, females = m_info.index[m_info.sex.eq('male')], m_info.index[m_info.sex.eq('female')]


def lake_pseudobulk():
    import h5py
    from pseudospace.replication_rules import csr_group_sums

    def column(group, name):
        item = group[name]
        if isinstance(item, h5py.Group):
            categories = np.array([c.decode() if isinstance(c, bytes) else str(c) for c in item['categories'][:]])
            return pd.Categorical.from_codes(item['codes'][:], categories).astype(str)
        values = item[:]
        return values.astype(str) if values.dtype.kind in 'SO' else values

    with h5py.File(lake_file, 'r') as handle:
        obs = pd.DataFrame({c: column(handle['obs'], c) for c in ['subclass.l2', 'disease', 'region', 'tissue', 'sex', 'donor_id']})
        var_ids = handle['raw/var/_index'][:].astype(str)
        var_names = np.asarray(column(handle['raw/var'], 'feature_name'))
        keep = (obs['subclass.l2'].isin(['PT-S1', 'PT-S2', 'PT-S3']) & obs.disease.eq('normal')
                & obs.region.eq('C') & obs.tissue.eq('cortex of kidney'))
        label = (obs.donor_id + '|' + obs.sex + '|' + obs['subclass.l2'].str.replace('PT-', '', regex=False))[keep]
        groups, codes = np.unique(label.to_numpy(), return_inverse=True)
        raw = handle['raw/X']
        sums, totals, nuclei = csr_group_sums(raw['data'], raw['indices'], raw['indptr'], np.flatnonzero(keep.to_numpy()),
                                              codes, len(groups), len(var_ids), block_rows=5000)
    human_source = ad.read_h5ad(source_paths[expected['human'][0]], backed='r')
    human_var = human_source.var[['gene_ids']].copy()
    human_source.file.close()
    by_id = dict(zip(orthologs.human_symbol.map(human_var.gene_ids), orthologs.mouse_symbol))
    by_symbol = dict(zip(orthologs.human_symbol, orthologs.mouse_symbol))
    target = np.array([by_id.get(i, by_symbol.get(s, '')) for i, s in zip(var_ids, var_names)])
    mapped = target != ''
    frame = pd.DataFrame(sums[:, mapped], columns=target[mapped]).T.groupby(level=0).sum().T.reindex(columns=genes, fill_value=0.)
    return {'groups': groups.astype(str), 'nuclei': nuclei, 'totals': totals, 'sums': frame.to_numpy()}


lake = cached_payload('lake_pseudobulk', lake_pseudobulk, root=cache,
                      params={'labels': 'PT-S1,PT-S2,PT-S3', 'region': 'C, cortex of kidney', 'disease': 'normal'},
                      inputs={'lake': [lake_file.name, lake_file.stat().st_size], 'genes': digest(genes),
                              'orthologs': digest(orthologs)}, code=code)
lake_meta = pd.DataFrame([g.split('|') for g in lake['groups']], columns=['donor', 'sex', 'segment'])
lake_meta['nuclei'] = lake['nuclei']
nuclei_table = lake_meta.pivot_table(index='donor', columns='segment', values='nuclei', aggfunc='sum', fill_value=0)
lake_donors = nuclei_table.index[(nuclei_table.reindex(columns=['S1', 'S2', 'S3'], fill_value=0) >= MIN_CELLS_EXTERNAL).all(axis=1)]
sums_frame = pd.DataFrame(lake['sums'], columns=genes)


def lake_log_cpm(donor, parts):
    rows = lake_meta.index[lake_meta.donor.eq(donor) & lake_meta.segment.isin(parts)]
    return np.log2(sums_frame.loc[rows].sum() / lake['totals'][rows].sum() * 1e6 + 1)


lake_late = pd.DataFrame({d: lake_log_cpm(d, ['S3']) for d in lake_donors}).T
lake_early = pd.DataFrame({d: lake_log_cpm(d, ['S1', 'S2']) for d in lake_donors}).T
lake_whole = pd.DataFrame({d: lake_log_cpm(d, ['S1', 'S2', 'S3']) for d in lake_donors}).T
print(f'Census human {len(h_late)} donors · Lake {len(lake_donors)} donors · mice {len(males)} male, {len(females)} female')

### Replication scores per reference (notebook 37's `reference_analysis`)

In [ ]:
from pseudospace.pathway_decoys import welch_t

short = pd.Series(segment).str.replace('PT-', '', regex=False).to_numpy()
ours_means = {}
for name in names:
    for label, mask in (('S3', short == 'S3'), ('early', np.isin(short, ['S1', 'S2']))):
        ours_means[(name, label)] = np.asarray(Y[(specimen == name) & mask].mean(axis=0)).ravel()
ours_means = pd.DataFrame(ours_means, index=genes)
ours_contrast = pd.DataFrame({name: ours_means[(name, 'S3')] - ours_means[(name, 'early')] for name in names})
ours_raw = ours_contrast[expected['human']].mean(axis=1) - ours_contrast[expected['mouse']].mean(axis=1)
mouse_gradient_ours = ours_contrast[expected['mouse']].mean(axis=1)


def reference_inputs(human_late, human_early, mouse_late, mouse_early, mouse_donors):
    hd = human_late - human_early
    md = (mouse_late - mouse_early).loc[mouse_donors]
    level_h = pd.concat([human_late.mean(), human_early.mean()], axis=1).max(axis=1)
    level_m = pd.concat([mouse_late.loc[mouse_donors].mean(), mouse_early.loc[mouse_donors].mean()], axis=1).max(axis=1)
    keep = (covariates[['detection_mouse', 'detection_human']].min(axis=1).ge(.10) & level_h.ge(1) & level_m.ge(1)
            & hd.notna().all() & md.notna().all())
    g = keep.index[keep.to_numpy()]
    centred = ours_contrast - ours_contrast.loc[g].median()
    ours_c = centred[expected['human']].mean(axis=1) - centred[expected['mouse']].mean(axis=1)
    hc, mc = hd.sub(hd[g].median(axis=1), axis=0), md.sub(md[g].median(axis=1), axis=0)
    ext_raw = hd.mean() - md.mean()
    t = pd.Series(welch_t(hc[g].to_numpy(), mc[g].to_numpy())[0], index=g)
    slope_ours = np.polyfit(mouse_gradient_ours.loc[g], ours_raw.loc[g], 1)
    ours_slope_free = ours_raw - np.polyval(slope_ours, mouse_gradient_ours)
    beta_ext = np.polyfit(md[g].mean(), ext_raw.loc[g], 1)[0]
    t_free = pd.Series(welch_t(hc[g].to_numpy(), (1 + beta_ext) * mc[g].to_numpy())[0], index=g)
    strata_g = matching_strata(pd.DataFrame({'mean_expression': ((level_h + level_m) / 2).loc[g].to_numpy(),
                                             'detection': covariates.loc[g, ['detection_mouse', 'detection_human']].min(axis=1).to_numpy(),
                                             'coverage': covariates.loc[g, 'coverage'].to_numpy()}), bins=3)
    sets_g = {p: [x for x in v if x in set(g)] for p, v in gene_sets.items()}
    sets_g = {p: v for p, v in sets_g.items() if 5 <= len(v) < len(g)}
    residuals = pd.concat([hc[g] - hc[g].mean(), mc[g] - mc[g].mean()])
    return {'genes': g, 'signed': np.sign(ours_c.loc[g]) * t, 'slope-free': np.sign(ours_slope_free.loc[g]) * t_free,
            'strata': strata_g, 'sets': sets_g, 'residuals': residuals.to_numpy()}


slugs = {'Census cortex − male mouse (primary)': 'census_male', 'Lake/KPMP cortex − male mouse': 'lake_male',
         'Census cortex − female mouse': 'census_female', 'Lake/KPMP cortex − female mouse': 'lake_female'}
references = {'Census cortex − male mouse (primary)': (h_late, h_early, m_late, m_early, males),
              'Lake/KPMP cortex − male mouse': (lake_late, lake_early, m_late, m_early, males)}
reference_data = {label: reference_inputs(*spec) for label, spec in references.items()}
display(pd.DataFrame({label: {'informative genes': len(r['genes']), 'testable pathways': len(r['sets']),
                              'primary testable': sum(p in r['sets'] for p in primary_ids),
                              'relabel-called testable': sum(p in r['sets'] for p in relabel_only)}
                      for label, r in reference_data.items()}))

### Scored sets, guard against notebook 37, and the permutation runs

In [ ]:
from pseudospace.set_collection_null import (JointReplicationZ, collection_null, label_permutation_p,
                                             upper_p)


def scored_sets(ref):
    sets_g = ref['sets']
    pathways = {p: sets_g[p] for p in sorted(set(robust_ids) | set(relabel_all)) if p in sets_g}
    programs = {}
    for kind, assignment in (('primary', primary_programs), ('relabel', relabel_programs)):
        for program, ids in assignment.groupby(assignment).groups.items():
            union = sorted({g for p in ids if p in sets_g for g in sets_g[p]})
            if len(union) >= 5:
                programs[f'program:{kind}:{int(program)}'] = union
    return {**pathways, **programs}


def collections_of(sets_):
    keep = lambda ids: [p for p in ids if p in sets_]
    return {'primary (22)': keep(primary_ids), 'primary core (12)': keep(primary_core), 'robust (36)': keep(robust_ids),
            'relabel-called, not robust (21)': keep(relabel_only), 'relabel-called, all (28)': keep(relabel_all),
            'primary programs': [p for p in sets_ if p.startswith('program:primary:')],
            'relabel-called programs': [p for p in sets_ if p.startswith('program:relabel:')]}


primary_label = 'Census cortex − male mouse (primary)'
primary_ref = reference_data[primary_label]
primary_sets = scored_sets(primary_ref)
primary_scorer = JointReplicationZ(primary_ref['signed'], primary_ref['strata'], primary_sets, primary_ref['residuals'])
guard = pd.DataFrame({'exact moments': pd.Series(primary_scorer.z(), index=primary_scorer.ids).reindex(robust_ids),
                      'notebook 37': table_s2.loc[robust_ids, 'external_z']})
guard_diff = (guard['exact moments'] - guard['notebook 37']).abs()
print(f'Guard: robust-pathway replication z, exact moments vs notebook 37: median |Δ| {guard_diff.median():.3f}, '
      f'max {guard_diff.max():.3f}, r = {guard.corr().iloc[0, 1]:.4f}')
if guard_diff.max() > .3 or guard.corr().iloc[0, 1] < .99:
    raise ValueError('The replication score does not reproduce notebook 37.')



In [ ]:
def run_null(label, statistic, null):
    ref = reference_data[label]
    sets_ = scored_sets(ref)
    strata = ref['strata']
    if null == 'selection-matched':
        quintile = pd.qcut(T_spatial.loc[ref['genes']].rank(method='first'), 5, labels=False).to_numpy()
        strata = strata * 5 + quintile
    scorer = JointReplicationZ(ref[statistic], strata, sets_, ref['residuals'])
    observed, draws = collection_null(scorer, collections_of(sets_), strata, N_PERM, SEED)
    return {'observed': observed.to_numpy(), 'draws': draws.to_numpy(), 'collections': np.array(list(observed.index), dtype=str),
            'set_ids': np.array(scorer.ids, dtype=str), 'set_z': scorer.z()}


runs = {}
for label in reference_data:
    for statistic, null in (('signed', 'matched'), ('signed', 'selection-matched'), ('slope-free', 'matched')):
        ref = reference_data[label]
        runs[(label, statistic, null)] = cached_payload(
            f'collection_null_{slugs[label]}_{statistic}_{null}',
            lambda label=label, statistic=statistic, null=null: run_null(label, statistic, null), root=cache,
            params={'n_perm': N_PERM, 'seed': SEED, 'label': label, 'statistic': statistic, 'null': null},
            inputs={'statistic': digest(ref[statistic]), 'strata': ref['strata'], 'T_spatial': digest(T_spatial),
                    'sets': digest(sorted(scored_sets(ref).items())), 'collections': digest(sorted(collections_of(scored_sets(ref)).items()))},
            code=code)



rows = []
for (label, statistic, null), run in runs.items():
    collections = list(run['collections'])
    observed = pd.Series(run['observed'], index=collections)
    draws = pd.DataFrame(run['draws'], columns=collections)
    set_z = pd.Series(run['set_z'], index=run['set_ids'])
    row = {'reference': label, 'statistic': statistic, 'null': null}
    for name in collections:
        row[f'mean z, {name}'] = observed[name]
    for name in ('primary (22)', 'primary core (12)', 'primary programs', 'relabel-called, not robust (21)'):
        row[f'p_random, {name}'] = upper_p(observed[name], draws[name])
    for first in ('primary (22)', 'primary core (12)'):
        for comparator, tag in (('relabel-called, not robust (21)', '21'), ('relabel-called, all (28)', '28')):
            diff = observed[first] - observed[comparator]
            row[f'D {first} vs relabel {tag}'] = diff
            row[f'p_relabel {first} vs {tag}'] = upper_p(diff, draws[first] - draws[comparator])
    primary_prog_z = set_z[[i for i in set_z.index if i.startswith('program:primary:')]]
    relabel_prog_z = set_z[[i for i in set_z.index if i.startswith('program:relabel:')]]
    row['programs (primary / relabel)'] = f'{len(primary_prog_z)} / {len(relabel_prog_z)}'
    row['p_relabel programs (label permutation)'] = label_permutation_p(primary_prog_z, relabel_prog_z, seed=SEED)[0]
    row['p_relabel programs (gene relabeling)'] = upper_p(observed['primary programs'] - observed['relabel-called programs'],
                                                         draws['primary programs'] - draws['relabel-called programs'])
    rows.append(row)
replication = pd.DataFrame(rows)
replication.to_csv(tables / 'table_replication_primary_overlap_null.csv', index=False)
display(replication.T)
per_set = pd.concat([pd.DataFrame({'set': run['set_ids'], 'z_joint': run['set_z']}).assign(reference=label, statistic=statistic)
                     for (label, statistic, null), run in runs.items() if null == 'matched'], ignore_index=True)
per_set['name'] = per_set.set.map(lambda s: pathway_names.get(s, s))
per_set['primary'], per_set['primary core'] = per_set.set.isin(primary_ids), per_set.set.isin(primary_core)
per_set.to_csv(tables / 'replication_z_per_set.csv', index=False)


def decide(row, first):
    beats_random = row[f'p_random, {first}'] <= .01
    beats_relabel = row[f'p_relabel {first} vs 21'] <= .05 and row['p_relabel programs (label permutation)'] <= .05
    return 'specific' if beats_random and beats_relabel else 'better than random only' if beats_random else 'not replicated'


primary_rows = replication[replication.statistic.eq('signed') & replication.null.eq('matched')].set_index('reference')
decisions = pd.DataFrame({first: {label: decide(row, first) for label, row in primary_rows.iterrows()}
                          for first in ('primary (22)', 'primary core (12)')})
decisions.to_csv(tables / 'decision_replication_primary.csv')
display(decisions)

## 6 · Conventional-only pathways (v0.1 minor item 20)

Notebook 37's claim: the conventional-only species pathways replicate their **average** human −
mouse difference better than other pathways (Mann–Whitney p = 7.2 × 10⁻⁴). These are pathways called
by whole-PT or S1/S2/S3 pseudobulk GSEA but not among the 60 `T_spatial` candidates.

The inputs are rebuilt exactly as notebook 37 builds them:
- specimen × segment pseudobulk counts, keeping genes with ≥ 10 counts in ≥ 2 specimens in every
  segment;
- DESeq2 `~species`, whole PT and per segment;
- signed preranked GSEA (gseapy multilevel, seed 12), with BH within whole PT and within segments;
- the external test: per-donor whole-PT log2 CPM, median-centred, Welch t of human against male
  mouse donors, signed by our whole-PT log2 fold change, then the matched rank-AUC with a
  donor-level VIF.

The guard requires 184 pathways and the Mann–Whitney p within a factor of 2 of notebook 37's.
Then the same overlap-preserving null tests:
- (i) the 184 as a collection;
- (ii) D = mean z(184) − mean z(other testable pathways), with both relabeled jointly.

In [ ]:
import gseapy as gp
from importlib.metadata import version
from scipy.stats import mannwhitneyu
from pseudospace.conventional_pt import deseq2_contrast

eligible_mask = universe.eligible.to_numpy()
raw_counts = sparse.csr_matrix(adata.layers['counts'], dtype=float)[:, eligible_mask]
segments37 = ('PT-S1', 'PT-S2', 'PT-S3')
blocks = {label: pd.DataFrame(np.vstack([np.asarray(raw_counts[(specimen == n) & (segment == label)].sum(axis=0)).ravel()
                                         for n in names]), index=names, columns=genes) for label in segments37}
count_support = np.logical_and.reduce([blocks[s].ge(10).sum(axis=0).ge(2).to_numpy() for s in segments37])
count_genes = pd.Index(genes)[count_support]
count_sets = {p: [g for g in v if g in set(count_genes)] for p, v in gene_sets.items()}
count_sets = {p: v for p, v in count_sets.items() if 10 <= len(v) <= 300}
whole_counts = sum(blocks[s] for s in segments37)[count_genes].astype(np.int64)
conventional_code = digest([code, 'conventional-v1', version('pydeseq2'), version('gseapy')])


def prerank(stat, sets, weight):
    ranking = stat.dropna().rename_axis('gene').reset_index(name='stat').sort_values(
        ['stat', 'gene'], ascending=[False, True], kind='stable')
    table = gp.prerank(rnk=ranking, gene_sets=sets, min_size=10, max_size=300, weight=weight,
                       method='multilevel', seed=GSEA_SEED, threads=4, outdir=None, no_plot=True).res2d
    table = table.rename(columns={'Term': 'pathway_id'})
    return pd.DataFrame({'pathway_id': table.pathway_id, 'NES': pd.to_numeric(table.NES), 'p': pd.to_numeric(table['NOM p-val'])})


def species_deseq2():
    metadata = pd.DataFrame({'species': ['human' if n in expected['human'] else 'mouse' for n in names]}, index=names)
    scopes = [('whole_PT', whole_counts)] + [(s, blocks[s][count_genes].astype(np.int64)) for s in segments37]
    return pd.concat([deseq2_contrast(block, metadata, design='~species', contrast=('species', 'human', 'mouse')).assign(scope=scope)
                      for scope, block in scopes], ignore_index=True)


count_inputs = {'counts': digest([whole_counts] + [blocks[s][count_genes] for s in segments37]), 'sets': digest(sorted(count_sets.items()))}
species_de = cached_frame('deseq2_species', species_deseq2, root=cache, inputs=count_inputs, code=conventional_code)
species_gsea = cached_frame('signed_gsea_species', lambda: pd.concat(
    [prerank(rows_.set_index('gene').stat, count_sets, 1).assign(scope=scope) for scope, rows_ in species_de.groupby('scope', sort=True)],
    ignore_index=True), root=cache, params={'weight': 1, 'method': 'multilevel', 'seed': GSEA_SEED},
    inputs={'de': digest(species_de[['gene', 'stat', 'scope']]), 'sets': count_inputs['sets']}, code=conventional_code)
species_gsea['family'] = np.where(species_gsea.scope.eq('whole_PT'), 'whole_PT', 'segments')
species_gsea['q'] = species_gsea.groupby('family').p.transform(lambda p: multipletests(p, method='fdr_bh')[1])
whole_hits = set(species_gsea.loc[species_gsea.family.eq('whole_PT') & species_gsea.q.le(ALPHA), 'pathway_id'])
segment_hits = set(species_gsea[species_gsea.family.eq('segments')].groupby('pathway_id').q.min().loc[lambda q: q.le(ALPHA)].index)
conventional_species = whole_hits | segment_hits
spatial_species = species37[species37.statistic.eq('T_spatial')]
candidates = set(spatial_species.loc[spatial_species.effect.gt(0) & spatial_species.q_joint.le(ALPHA), 'pathway_id'])
whole_lfc = species_de[species_de.scope.eq('whole_PT')].set_index('gene').log2FoldChange
s1_37 = pd.read_csv(upstream37 / 'table_S1_strategy_specificity.csv', index_col=0)
print(f'Conventional species calls: whole-PT {len(whole_hits)} (notebook 37: {int(s1_37.loc["1 · Whole-PT pseudobulk + signed GSEA", "species_calls"])}), '
      f'segments {len(segment_hits)} (notebook 37: {int(s1_37.loc["2 · S1/S2/S3 pseudobulk + signed GSEA", "species_calls"])}); '
      f'T_spatial candidates {len(candidates)}')


def average_level_inputs(label, human_whole, mouse_whole, mouse_donors):
    info = reference_data[label]['genes']
    hw = human_whole.sub(human_whole[info].median(axis=1), axis=0)
    mw = mouse_whole.loc[mouse_donors].sub(mouse_whole.loc[mouse_donors, info].median(axis=1), axis=0)
    t_main = pd.Series(welch_t(hw[info].to_numpy(), mw[info].to_numpy())[0], index=info)
    sets_ = {p: [x for x in v if x in set(info)] for p, v in gene_sets.items()}
    sets_ = {p: v for p, v in sets_.items() if 5 <= len(v) < len(info)}
    residuals = pd.concat([hw[info] - hw[info].mean(), mw[info] - mw[info].mean()])
    statistic = np.sign(whole_lfc.reindex(info).fillna(0)) * t_main
    strata = matching_strata(pd.DataFrame({'mean_expression': ((human_whole[info].mean() + mouse_whole.loc[mouse_donors, info].mean()) / 2).to_numpy(),
                                           'detection': covariates.loc[info, ['detection_mouse', 'detection_human']].min(axis=1).to_numpy(),
                                           'coverage': covariates.loc[info, 'coverage'].to_numpy()}), bins=3)
    return {'statistic': statistic, 'sets': sets_, 'residuals': residuals.to_numpy(), 'strata': strata}


average_refs = {primary_label: average_level_inputs(primary_label, h_whole, m_whole, males),
                'Lake/KPMP cortex − male mouse': average_level_inputs('Lake/KPMP cortex − male mouse', lake_whole, m_whole, males)}


def conventional_null(label):
    ref = average_refs[label]
    scorer = JointReplicationZ(ref['statistic'], ref['strata'], ref['sets'], ref['residuals'])
    conventional_only = sorted((conventional_species - candidates) & set(ref['sets']))
    others = [p for p in ref['sets'] if p not in set(conventional_only)]
    observed, draws = collection_null(scorer, {'conventional-only': conventional_only, 'other': others}, ref['strata'], N_PERM, SEED)
    return {'observed': observed.to_numpy(), 'draws': draws.to_numpy(), 'set_ids': np.array(scorer.ids, dtype=str),
            'set_z': scorer.z(), 'conventional_only': np.array(conventional_only, dtype=str)}


conventional_runs = {label: cached_payload(f'conventional_only_null_{slugs[label]}', lambda label=label: conventional_null(label), root=cache,
                                           params={'n_perm': N_PERM, 'seed': SEED},
                                           inputs={'statistic': digest(average_refs[label]['statistic']), 'strata': average_refs[label]['strata'],
                                                   'sets': digest(sorted(average_refs[label]['sets'].items())),
                                                   'conventional': sorted(conventional_species), 'candidates': sorted(candidates)}, code=code)
                     for label in average_refs}
saved_conventional = pd.read_csv(upstream37 / 'external_average_level_conventional_only.csv', index_col=0)['0']
conventional_rows = {}
for label, run in conventional_runs.items():
    z = pd.Series(run['set_z'], index=run['set_ids'])
    conventional_only = list(run['conventional_only'])
    others = z.index.difference(conventional_only)
    draws = pd.DataFrame(run['draws'], columns=['conventional-only', 'other'])
    observed = pd.Series(run['observed'], index=['conventional-only', 'other'])
    conventional_rows[label] = {
        'conventional-only pathways tested': len(conventional_only),
        'median z, conventional-only': float(z.loc[conventional_only].median()), 'median z, other': float(z.loc[others].median()),
        'Mann–Whitney p (overlap-naive, as notebook 37)': mannwhitneyu(z.loc[conventional_only], z.loc[others], alternative='greater').pvalue,
        'mean z, conventional-only': observed['conventional-only'], 'mean z, other': observed['other'],
        'p, conventional-only vs random collections': upper_p(observed['conventional-only'], draws['conventional-only']),
        'D = mean(conventional-only) − mean(other)': observed['conventional-only'] - observed['other'],
        'p, D under joint relabeling': upper_p(observed['conventional-only'] - observed['other'], draws['conventional-only'] - draws['other'])}
conventional_table = pd.DataFrame(conventional_rows)
conventional_table.to_csv(tables / 'conventional_only_overlap_null.csv')
display(conventional_table)
primary_conv = conventional_table[primary_label]
reproduced = (primary_conv['conventional-only pathways tested'] == int(saved_conventional['conventional-only species pathways tested'])
              and .5 <= primary_conv['Mann–Whitney p (overlap-naive, as notebook 37)'] / saved_conventional['Mann–Whitney p, conventional-only > other'] <= 2)
CONVENTIONAL_DECISION = ('cannot be reproduced' if not reproduced else
                         'group claim stands with overlap respected' if primary_conv['p, D under joint relabeling'] <= .05 else
                         'group claim not supported once overlap is respected')
print('Guard (184 pathways, Mann–Whitney within a factor of 2):', reproduced, '→', CONVENTIONAL_DECISION)

## 7 · Run record

In [ ]:
record = {'notebook_logic_version': NOTEBOOK_LOGIC_VERSION, 'n_perm': N_PERM, 'seed': SEED,
          'primary': len(primary_ids), 'primary_core': len(primary_core), 'step_screen_checks': checks_,
          'replication_decisions': decisions.to_dict(), 'conventional_only_decision': CONVENTIONAL_DECISION}
(output / 'run_record.json').write_text(json.dumps(record, indent=2, default=str))
print(json.dumps(record, indent=2, default=str))